# DEST — Kaggle Minimal (sin pip, no mata kernel)

**No reinstala torch — usa el de Kaggle (T4 sm_75)**

* Clona DEST fix (`lexsort` 45k únicos) y lo añade a `sys.path` sin `pip install`
* 20 limpias `stochastic` vs `collatz_v3` seeds 300–309, CIFAR-10, ~70 min, guarda en `/kaggle/working`
* Para 20h usa el `SeedPool` comentado abajo


In [ ]:
# 0. Setup sin pip — usa torch de Kaggle
import os, sys, subprocess
print("🔧 Setup sin pip...")
if os.path.exists("DEST"):
    print("DEST ya existe")
else:
    subprocess.check_call(["git","clone","https://github.com/starlyn2010/DEST.git"])
    print("✅ Clonado")

# NO hacer pip install — solo añadir a path
if "DEST/src" not in sys.path:
    sys.path.insert(0, "DEST/src")
print("sys.path:", sys.path[:3])

# Alias dest_lib -> dest para compatibilidad con runner
import dest
sys.modules["dest_lib"] = dest
for sub in ["config","samplers","models","datasets","runner","metrics","reproducibility"]:
    try:
        m = __import__(f"dest.{sub}", fromlist=[sub])
        sys.modules[f"dest_lib.{sub}"] = m
        print(f"✅ dest_lib.{sub}")
    except Exception as e:
        print(f"warn {sub}: {e}")

import torch
print("\nCUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("torch", torch.__version__)
# Test rápido
import torch.nn as nn
try:
    m=nn.Conv2d(3,16,3,padding=1).cuda()
    x=torch.randn(2,3,32,32).cuda()
    y=m(x)
    print("✅ Conv2d CUDA OK", y.shape)
except Exception as e:
    print("❌ Conv prueba falló:", e)
    print("   Si falla, cambia a T4 x2 en Settings")


In [ ]:
# 1. Config 20 limpias
from dest_lib.config import get_config
config=get_config("PAPER")
config["datasets"]=["CIFAR10"]
config["samplers"]=["stochastic","collatz_v3"]
config["seeds"]=list(range(300,310))
config["epochs"]=15
config["batch_size"]=128
config["lr"]=0.01
config["lr_schedule"]="cosine"
config["output_dir"]="/kaggle/working/dest_kaggle_clean"
config["val_fraction"]=0.1
config["verbose"]=True
import os, json
os.makedirs(config["output_dir"], exist_ok=True)
print(f"Output: {config['output_dir']} ({len(os.listdir(config['output_dir']))} existentes)")
print(f"Total: {len(config['seeds'])*len(config['samplers'])} runs")


In [ ]:
# 2. Ejecutar — reanudable
import os, time, json, glob
from dest_lib.runner import ExperimentRunner
runner=ExperimentRunner(config)
# Contar hechos
done=sum(1 for s in config["seeds"] for sa in config["samplers"] if os.path.exists(os.path.join(config["output_dir"], f"CIFAR10_{sa}_{sa}_seed_{s}.json")) and json.load(open(os.path.join(config["output_dir"], f"CIFAR10_{sa}_{sa}_seed_{s}.json"))).get("status")=="COMPLETE")
print(f"Ya completados: {done}/20, faltan {20-done}")
start_all=time.time()
for seed in config["seeds"]:
    for sampler_name in config["samplers"]:
        out_file=os.path.join(config["output_dir"], f"CIFAR10_{sampler_name}_{sampler_name}_seed_{seed}.json")
        if os.path.exists(out_file):
            try:
                j=json.load(open(out_file))
                if j.get("status")=="COMPLETE" and len(j.get("test_accs",[]))==15:
                    print(f"⏭️ {sampler_name} {seed} ya está ({j['final_test_acc']:.2f}%)")
                    continue
                else: os.remove(out_file)
            except: os.remove(out_file) if os.path.exists(out_file) else None
        print(f"\n▶️ {sampler_name} seed {seed}")
        r=runner.run_single_seed(exp_id=f"CIFAR10_{sampler_name}", sampler_name=sampler_name, seed=seed, dataset="CIFAR10")
        print(f"✅ {sampler_name} {seed}: {r.final_test_acc:.2f}% en {r.total_runtime_seconds/60:.1f} min")
print(f"\n✅ Todo completo en {(time.time()-start_all)/60:.1f} min")


In [ ]:
# 3. Resumen y zip
import glob, json, numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt
pattern="/kaggle/working/dest_kaggle_clean/*.json"
files=[f for f in glob.glob(pattern) if "sampler_name" in json.load(open(f))]
print(f"JSONs válidos: {len(files)}/20")
if files:
    groups=defaultdict(list)
    for f in files:
        j=json.load(open(f)); groups[j["sampler_name"]].append(j)
    for s in ["stochastic","collatz_v3"]:
        arr=[j["final_test_acc"] for j in groups[s]]
        if arr: print(f"{s:12s}: {np.mean(arr):.2f} ±{np.std(arr,ddof=1):.2f} n={len(arr)}")
    if "stochastic" in groups and "collatz_v3" in groups:
        stoch={j["seed"]:j["final_test_acc"] for j in groups["stochastic"]}
        v3={j["seed"]:j["final_test_acc"] for j in groups["collatz_v3"]}
        common=sorted(set(stoch)&set(v3))
        diffs=[v3[s]-stoch[s] for s in common]
        if diffs:
            from scipy import stats
            t,p=stats.ttest_rel([v3[s] for s in common],[stoch[s] for s in common])
            print(f"V3 vs stoch: diff {np.mean(diffs):+.3f} p={p:.4f} gana {sum(d>0 for d in diffs)}/{len(diffs)}")
    plt.figure(figsize=(7,3))
    for s in ["stochastic","collatz_v3"]:
        if s not in groups: continue
        arr=np.array([j["test_accs"] for j in groups[s]])
        plt.plot(range(1,16), arr.mean(0), label=s)
        plt.fill_between(range(1,16), arr.mean(0)-arr.std(0,ddof=1), arr.mean(0)+arr.std(0,ddof=1), alpha=0.15)
    plt.title("CIFAR-10 300–309 Limpio (fix 45k únicos)"); plt.xlabel("Época"); plt.ylabel("Test acc %")
    plt.legend(); plt.grid(alpha=0.3)
    plt.savefig("/kaggle/working/dest_kaggle_clean/curvas.png", dpi=200, bbox_inches="tight")
    plt.show()

# Zip para Datasets versionable
import shutil, os
zipname="/kaggle/working/resultados_Kaggle_Limpio_300_309"
shutil.make_archive(zipname, 'zip', "/kaggle/working/dest_kaggle_clean")
print(f"✅ ZIP {os.path.getsize(zipname+'.zip')/1e6:.2f} MB en {zipname}.zip")
print("En Kaggle: Output → /kaggle/working/resultados_Kaggle_Limpio_300_309.zip → Add to Datasets")
